# Dow Jones 30 — Dataset Creation

All indicator combinations for the RL state representation research.

**Pipeline:** Raw OHLCV → FeatureEngineer → compound triggers → filter → save CSV

**Output:** `data/dow/dow_chlovtd_{variant}_{start}_to_{end}.csv`

In [ ]:
import os, pandas as pd
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from finrl.config import INDICATORS
from pathlib import Path
from utils.config import (
    DOW_30_TICKER,
    INDICATORS_BASE, INDICATORS_MVX2ALL, INDICATORS_MVXALL, INDICATORS_BOLLXALL, INDICATORS_MVX3
)
from utils.common import filter_x_keep_all_companies

In [3]:
START_DATE = '2009-01-01'
END_DATE   = '2026-01-01'
DATE_TAG   = f"{START_DATE}_to_{END_DATE}"
DATA_DIR   = Path(os.environ['FINRL_PATH']+"/data")
# TODO: SUBDIR = 'dow'

## 1. Raw Data

In [4]:
RAW_PATH = f"{DATA_DIR}/dow/dow_chovtd_{DATE_TAG}.csv"
if not os.path.exists(RAW_PATH):
    df_raw = YahooDownloader(start_date=START_DATE, end_date=END_DATE,
                             ticker_list=DOW_30_TICKER).fetch_data()
    os.makedirs(DATA_DIR, exist_ok=True)
    df_raw.to_csv(RAW_PATH, index=False)
    print(f"Downloaded and saved → {RAW_PATH}")
else:
    df_raw = pd.read_csv(RAW_PATH)
    print(f"Loaded from {RAW_PATH}")
print(f"Shape: {df_raw.shape}  |  Tickers: {df_raw['tic'].nunique()}  |  Dates: {df_raw['date'].nunique()}")
df_raw.head()

Loaded from /home/main/Programming/finrl/data/dow/dow_chovtd_2009-01-01_to_2026-01-01.csv
Shape: (128280, 8)  |  Tickers: 30  |  Dates: 4276


,date,close,high,low,open,volume,tic,day
0,2009-01-02,2.719142,2.727833,2.551650,2.573223,746015200,AAPL,4
1,2009-01-02,39.900166,39.961041,39.061443,39.629609,6547900,AMGN,4
2,2009-01-02,2.718000,2.726500,2.553500,2.567500,145928000,AMZN,4
3,2009-01-02,14.821135,14.966816,14.108064,14.238410,10955700,AXP,4
4,2009-01-02,33.941090,34.173615,32.088393,32.103395,7010200,BA,4


## 2. Feature Engineering

In [14]:
# All-in-one: FinRL default indicators + vix + turbulence
fe_all = FeatureEngineer(use_technical_indicator=True,
                         tech_indicator_list=INDICATORS,
                         use_vix=True, use_turbulence=True)
processed_all = fe_all.preprocess_data(df_raw)
print(f"all: {processed_all.shape}  cols: {list(processed_all.columns)}")

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (4275, 8)
Successfully added vix
Successfully added turbulence index
all: (128250, 18)  cols: ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma', 'vix', 'turbulence']


In [7]:
INDICATOR_SETS = INDICATORS_BASE.copy() 

INDICATOR_SETS = { # shorter for some that I forgot
   # 'mvx2all': INDICATORS_MVX2ALL,
   # 'mvxall': INDICATORS_MVXALL, 
   # 'bollxall':INDICATORS_BOLLXALL
    'mvx3': INDICATORS_MVX3 
     
}

processed = {}
for name, indicators in INDICATOR_SETS.items():
    if name in ['mvx2all', 'mvxall', 'bollxall']: # add turbulence and vix for the "all" sets
        fe = FeatureEngineer(use_technical_indicator=True,
                         tech_indicator_list=indicators,
                         use_vix=True, use_turbulence=True)
    else:
        fe = FeatureEngineer(use_technical_indicator=True,
                            tech_indicator_list=indicators,
                            use_vix=False, use_turbulence=False)
    processed[name] = fe.preprocess_data(df_raw)
    print(f"  {name:12s}: {processed[name].shape}  cols: {list(processed[name].columns)}")

Successfully added technical indicators
  mvx3        : (128280, 12)  cols: ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day', 'close_10_sma', 'close_30_sma', 'close_10_sma_xu_close_30_sma', 'close_10_sma_xd_close_30_sma']


## 3. Compound Triggers

Post-FeatureEngineer logic that combines base indicators into composite signals.

In [8]:
# 3MVX1: EMA-5 crosses EMA-10 while EMA-10 > EMA-50
df = processed['3mvx1']
df['ema_xu_5_10_50'] = df['close_5_ema_xu_close_10_ema'] & (df['close_10_ema'] > df['close_50_ema'])
df['ema_xd_5_10_50'] = df['close_5_ema_xd_close_10_ema'] & (df['close_10_ema'] < df['close_50_ema'])
print(f"3mvx1 triggers  up: {df['ema_xu_5_10_50'].sum()}  down: {df['ema_xd_5_10_50'].sum()}")

3mvx1 triggers  up: 2832  down: 1433


In [9]:
# 3MVX2: EMA-5 crosses EMA-50 while 5 >= 10 >= 50 (full alignment)
df = processed['3mvx2']
df['ema_xu_5_10_50'] = (df['close_5_ema_xu_close_50_ema'] &
                        (df['close_5_ema'] >= df['close_10_ema']) &
                        (df['close_10_ema'] >= df['close_50_ema']))
df['ema_xd_5_10_50'] = (df['close_5_ema_xd_close_50_ema'] &
                         (df['close_5_ema'] <= df['close_10_ema']) &
                         (df['close_10_ema'] <= df['close_50_ema']))
print(f"3mvx2 triggers  up: {df['ema_xu_5_10_50'].sum()}  down: {df['ema_xd_5_10_50'].sum()}")

3mvx2 triggers  up: 504  down: 434


In [10]:
# BOLLRSIX1: mean-reversion (lower band touch + RSI oversold / upper band touch + RSI overbought)
df = processed['bollrsix1']
df['bb_xrsi_long'] = df['close_xu_boll_lb'] & (df['rsi_14'] < 35)
df['bb_rsi_short']  = df['close_xd_boll_ub'] & (df['rsi_14'] > 65)
print(f"bollrsix1 triggers  long: {df['bb_xrsi_long'].sum()}  short: {df['bb_rsi_short'].sum()}")

bollrsix1 triggers  long: 884  short: 1934


In [11]:
# BOLLRSIX2: breakout (upper band break + RSI strong + expanding volatility)
# groupby('tic') prevents rolling mean from bleeding across tickers
df = processed['bollrsix2']
df['bb_width'] = (df['boll_ub'] - df['boll_lb']) / df['close']
bb_width_mean = df.groupby('tic')['bb_width'].transform(lambda x: x.rolling(20).mean())
volatility_expanding = df['bb_width'] > bb_width_mean
df['bb_breakout_long']  = df['close_xu_boll_ub'] & (df['rsi_14'] > 60) & volatility_expanding
df['bb_breakout_short'] = df['close_xd_boll_lb'] & (df['rsi_14'] < 35) & volatility_expanding
print(f"bollrsix2 triggers  long: {df['bb_breakout_long'].sum()}  short: {df['bb_breakout_short'].sum()}")

bollrsix2 triggers  long: 1625  short: 999


## 4. Filtered Datasets (all companies on trigger dates)

In [8]:
TRIGGER_VARIANTS = ['mvx', 'mvx2', 'bollx', 'bollmvx', 'bollmvx2',
                    '3mvx1', '3mvx2', 'bollrsix1', 'bollrsix2']

TRIGGER_VARIANTS = ['mvx2all', 'mvxall', 'bollxall']

TRIGGER_VARIANTS = ["mvx3"]

for name in TRIGGER_VARIANTS:
    processed[f'flt_{name}'] = filter_x_keep_all_companies(processed[name])
    full = processed[name].shape[0]
    flt  = processed[f'flt_{name}'].shape[0]
    pct  = flt / full * 100
    print(f"  flt_{name:10s}: {flt:>7,} / {full:>7,} rows ({pct:.1f}%)  "
          f"tickers: {processed[f'flt_{name}']['tic'].nunique()}")

  flt_mvx3      :  81,750 / 128,280 rows (63.7%)  tickers: 30


## 5. Export to CSV

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)

# Save all-in-one
path = f"{DATA_DIR}/dow/dow_chlovtd_all_{DATE_TAG}.csv"
processed_all.to_csv(path, index=False)
print(f"  {path}  {processed_all.shape}")

# Save all custom datasets (full + filtered)
for name, df in processed.items():
    path = f"{DATA_DIR}/dow/dow_chlovtd_{name}_{DATE_TAG}.csv"
    df.to_csv(path, index=False)
    print(f"  {path}  {df.shape}")

  /home/main/Programming/finrl/data/dow/dow_chlovtd_mvx3_2009-01-01_to_2026-01-01.csv  (128280, 12)
  /home/main/Programming/finrl/data/dow/dow_chlovtd_flt_mvx3_2009-01-01_to_2026-01-01.csv  (81750, 12)


## 6. Summary

In [ ]:
all_dfs = {'all': processed_all, **processed}
summary = pd.DataFrame([
    {'dataset': name, 'rows': df.shape[0], 'cols': df.shape[1],
     'dates': df['date'].nunique(), 'tickers': df['tic'].nunique()}
    for name, df in all_dfs.items()
])
summary

## 7. Load All Datasets from CSV

In [16]:
# Re-load all chlovtd datasets from CSV (run this cell to skip recomputation above)
ALL_NAMES = ['all', 'mv', 'mv2', 'mvx', 'mvx2', 'boll', 'bollx',
             'bollmvx', 'bollmvx2', '3mvx1', '3mvx2', 'bollrsix1', 'bollrsix2',
             'flt_mvx', 'flt_mvx2', 'flt_bollx',
             'flt_bollmvx', 'flt_bollmvx2',
             'flt_3mvx1', 'flt_3mvx2',
             'flt_bollrsix1', 'flt_bollrsix2',
             'flt_mvx2all', 'flt_mvxall', 'flt_bollxall'
             'mvx2all', 'mvxall', 'bollxall']
ALL_NAMES = [
            'mvx3', 'flt_mvx3']

loaded = {}
for name in ALL_NAMES:
    if name == 'all':
        path = f"{DATA_DIR}/dow/dow_chlovtd_all_{DATE_TAG}.csv"
    else:
        path = f"{DATA_DIR}/dow_chlovtd_{name}_{DATE_TAG}.csv"
    if os.path.exists(path):
        loaded[name] = pd.read_csv(path)
        print(f"  {name:25s}: {loaded[name].shape}  ← {path}")
    else:
        print(f"  {name:25s}: NOT FOUND — {path}")

  mvx3                     : NOT FOUND — /home/main/Programming/finrl/data/dow_chlovtd_mvx3_2009-01-01_to_2026-01-01.csv
  flt_mvx3                 : NOT FOUND — /home/main/Programming/finrl/data/dow_chlovtd_flt_mvx3_2009-01-01_to_2026-01-01.csv


## 8. Compact Datasets — `cvtd` (drop open / high / low)

Remove the raw OHLC columns `open`, `high`, `low` to produce leaner datasets where the state contains only `close`, `volume`, the technical indicators, and `day`.

In [22]:
DROP_COLS = ['open', 'high', 'low']

def to_cvtd(df: pd.DataFrame) -> pd.DataFrame:
    """Drop open/high/low columns; keep everything else."""
    return df.drop(columns=[c for c in DROP_COLS if c in df.columns])

# Build cvtd variants from `loaded` (or fall back to `processed` / `processed_all`)
source = loaded if loaded else {**{'all': processed_all}, **processed}
#source = processed

cvtd = {}
for name, df in source.items():
    cvtd[name] = to_cvtd(df)

# Save
os.makedirs(f"{DATA_DIR}/dow", exist_ok=True)
for name, df in cvtd.items():
    if name == 'all':
        path = f"{DATA_DIR}/dow/dow_cvtd_all_{DATE_TAG}.csv"
    else:
        path = f"{DATA_DIR}/dow/dow_cvtd_{name}_{DATE_TAG}.csv"
    df.to_csv(path, index=False)
    print(f"  {name:25s}: {df.shape}  cols: {list(df.columns)}")
    print(f"  {'':25s}  → {path}")

  all                      : (128250, 15)  cols: ['date', 'close', 'volume', 'tic', 'day', 'macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma', 'vix', 'turbulence']
                             → /home/main/Programming/finrl/data/dow/dow_cvtd_all_2009-01-01_to_2026-01-01.csv
  mvx3                     : (128280, 9)  cols: ['date', 'close', 'volume', 'tic', 'day', 'close_10_sma', 'close_30_sma', 'close_10_sma_xu_close_30_sma', 'close_10_sma_xd_close_30_sma']
                             → /home/main/Programming/finrl/data/dow/dow_cvtd_mvx3_2009-01-01_to_2026-01-01.csv
  flt_mvx3                 : (81750, 9)  cols: ['date', 'close', 'volume', 'tic', 'day', 'close_10_sma', 'close_30_sma', 'close_10_sma_xu_close_30_sma', 'close_10_sma_xd_close_30_sma']
                             → /home/main/Programming/finrl/data/dow/dow_cvtd_flt_mvx3_2009-01-01_to_2026-01-01.csv


## 9. Combined Dual-MA Cross — `dbmvx`

Merge `mvx` (EMA-5 / SMA-20 + cross triggers) and `mvx2` (SMA-30 / SMA-60 + cross triggers) into one dataset with all 8 indicator columns.

In [18]:
_BASE = ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day']

# Use loaded CSVs if available, otherwise fall back to in-memory processed dicts
source = loaded if loaded else {**{'all': processed_all}, **processed}

In [ ]:

_mvx_extra  = [c for c in source['mvx'].columns  if c not in _BASE]
_mvx2_extra = [c for c in source['mvx2'].columns if c not in _BASE]

df_dbmvx = source['mvx'].merge(
    source['mvx2'][_BASE + _mvx2_extra],
    on=_BASE
)
df_dbmvx_flt = filter_x_keep_all_companies(df_dbmvx)

print(f"dbmvx         : {df_dbmvx.shape}  cols: {list(df_dbmvx.columns)}")
print(f"flt_dbmvx: {df_dbmvx_flt.shape}")

# Save chlovtd (full OHLCV)
os.makedirs(f"{DATA_DIR}/dow", exist_ok=True)
for name, df in [('dbmvx', df_dbmvx), ('flt_dbmvx', df_dbmvx_flt)]:
    path = f"{DATA_DIR}/dow/dow_chlovtd_{name}_{DATE_TAG}.csv"
    df.to_csv(path, index=False)
    print(f"  {path}  {df.shape}")

# Save cvtd (no open/high/low)
for name, df in [('dbmvx', df_dbmvx), ('flt_dbmvx', df_dbmvx_flt)]:
    path = f"{DATA_DIR}/dow/dow_cvtd_{name}_{DATE_TAG}.csv"
    to_cvtd(df).to_csv(path, index=False)
    print(f"  {path}  {to_cvtd(df).shape}")

KeyError: 'mvx'

## 10. Trigger-Only Variants — `_trg`

For every dataset that contains boolean trigger columns, produce a stripped version that keeps only:
- Base columns: `date`, `close`, `high`, `low`, `open`, `volume`, `tic`, `day`
- Boolean signal columns (cross-up / cross-down / compound triggers)

All continuous indicator values (SMA, EMA, Bollinger bands, RSI) are dropped.
Each variant is saved in both `chlovtd` (full OHLCV) and `cvtd` (no open/high/low) forms.

In [11]:
def to_trg(df: pd.DataFrame) -> pd.DataFrame:
    """Keep base OHLCV cols + boolean trigger columns only; drop continuous indicators."""
    base = [c for c in ['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day']
            if c in df.columns]
    trig = df.select_dtypes(include=['bool', 'boolean']).columns.tolist()
    return df[base + trig]

In [23]:
TRG_NAMES = [
    'mvx', 'mvx2', 'dbmvx',
    'bollx', 'bollmvx', 'bollmvx2',
    '3mvx1', '3mvx2',
    'bollrsix1', 'bollrsix2',
    'flt_mvx', 'flt_mvx2', 'flt_dbmvx',
    'flt_bollx', 'flt_bollmvx', 'flt_bollmvx2',
    'flt_3mvx1', 'flt_3mvx2',
    'flt_bollrsix1', 'flt_bollrsix2',
]

TRG_NAMES = [
    'mvx3',
    'flt_mvx3'
]

# Include dbmvx variants computed in section 9
#trg_source = {**source, 'dbmvx': df_dbmvx, 'flt_dbmvx': df_dbmvx_flt}
# MVX3
trg_source = {**source}

os.makedirs(f"{DATA_DIR}/dow", exist_ok=True)
for name in TRG_NAMES:
    df_trg = to_trg(trg_source[name])
    trg_name = f"{name}_trg"
    trig_cols = [c for c in df_trg.columns if c not in set(_BASE)]

    # chlovtd save (full OHLCV + triggers)
    path = f"{DATA_DIR}/dow/dow_chlovtd_{trg_name}_{DATE_TAG}.csv"
    df_trg.to_csv(path, index=False)

    # cvtd save (no open/high/low + triggers)
    path_cvtd = f"{DATA_DIR}/dow/dow_cvtd_{trg_name}_{DATE_TAG}.csv"
    to_cvtd(df_trg).to_csv(path_cvtd, index=False)

    print(f"  {trg_name:30s}: {df_trg.shape}  triggers: {trig_cols}")

  mvx3_trg                      : (128280, 10)  triggers: ['close_10_sma_xu_close_30_sma', 'close_10_sma_xd_close_30_sma']
  flt_mvx3_trg                  : (81750, 10)  triggers: ['close_10_sma_xu_close_30_sma', 'close_10_sma_xd_close_30_sma']
